# SSBU player detection — prediction overlay (one click)

Shared Drive: [bros-analysis](https://drive.google.com/drive/folders/12iT_dLsUOeFQ_UQBQsIWsnmD6ywvdPtl)

Runs your **already-trained** checkpoint over a split and writes raw predictions to
Drive, so the local review app can show *model predictions vs your labels* and rank the
worst disagreements first.

This does **not** retrain. Weights are only read.

1. Runtime -> Change runtime type -> **GPU**
2. Run the cells below (or **Runtime -> Run all**)

Logic lives on Drive in `scripts/predict_ssbu.py`. See `docs/label_review_app.md` in the
git repo for how to review the results.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/bros-analysis")
PREDICT_PY = ROOT / "scripts" / "predict_ssbu.py"

if not PREDICT_PY.exists():
    raise FileNotFoundError(
        f"Missing {PREDICT_PY}\n"
        "bros-analysis is not in My Drive. The folder is restricted: accept the\n"
        "maintainer's invite, then right-click it -> Organize -> Add shortcut to\n"
        "My Drive (named 'bros-analysis'). A folder only shared with you appears\n"
        "under Shared with me, which Colab cannot see."
    )
print("OK:", PREDICT_PY)

### Cell 2 - quick smoke test (25 images, about 30s)

Run this first to confirm the plumbing before committing to the full split.

In [ ]:
!python /content/drive/MyDrive/bros-analysis/scripts/predict_ssbu.py --split val --limit 25

### Cell 3 - full run over the held-out val split (772 images)

In [ ]:
!python /content/drive/MyDrive/bros-analysis/scripts/predict_ssbu.py --split val

### Optional - the training split (3085 images, slower)

Deliberately left as text, not a code cell, so **Run all** does not launch it.
Paste this into a new cell if you want it:

```
!python /content/drive/MyDrive/bros-analysis/scripts/predict_ssbu.py --split train
```


Results land in `exports/predictions/smash_player_colab_v2/<split>.jsonl` on Drive.

Point the local app at that folder to see the overlay:

```bash
python app/server.py --dataset .staging/ssbu_players_colab \
    --predictions "<drive>/exports/predictions/smash_player_colab_v2"
```